# Setup & Configuration


In [ ]:
import os, sys, time, gc, re, math, json, warnings
from collections import Counter, defaultdict
from itertools import product
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast as _autocast

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score
from scipy.special import softmax as _sci_softmax

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    p = torch.cuda.get_device_properties(0)
    print(f"  GPU: {p.name}, {p.total_memory/1e9:.1f} GB")

USE_BF16  = True
AMP_DTYPE = torch.bfloat16 if (USE_BF16 and DEVICE.type == 'cuda'
                               and torch.cuda.is_bf16_supported()) else torch.float16

def autocast_amp():
    return _autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda'))

torch.set_float32_matmul_precision('high')
print(f"AMP dtype: {AMP_DTYPE}")


In [ ]:
CFG = dict(
    DATA_DIR    = '/kaggle/input/competitions/smart-mcq-solver-challenge',
    OUTPUT_DIR  = '/kaggle/working',
    SEED        = 42,
    N_FOLDS     = 2,

    # LSTM
    LSTM_EMBED      = 256,
    LSTM_HIDDEN     = 384,
    LSTM_MAX_Q_LEN  = 80,
    LSTM_MAX_OPT_LEN= 25,
    BS_LSTM         = 256,

    # DeBERTa
    DEBERTA_MODEL = 'microsoft/deberta-v3-large',
    MAX_LEN_DEB   = 512,
    BS_DEB        = 4,

    # ELECTRA
    ELECTRA_MODEL = 'google/electra-large-discriminator',
    MAX_LEN_ELE   = 384,
    BS_ELE        = 4,
)

OPTS = ['A', 'B', 'C', 'D', 'E']
os.makedirs(CFG['OUTPUT_DIR'], exist_ok=True)
print("Config loaded.")



## 3 · Load competition data

In [ ]:
train = pd.read_csv(f"{CFG['DATA_DIR']}/train.csv")
test  = pd.read_csv(f"{CFG['DATA_DIR']}/test.csv")
print(f"train: {train.shape},  test: {test.shape}")


## 4 · Text normalization + GroupKFold

In [ ]:
_BOILERPLATE = [
    r"pick the best possible answer:?",
    r"\bcarefully\b\.?",
    r"among the listed options\.?",
    r"choose the (correct|best) (option|answer|response)\.?",
    r"select the (best|correct|most appropriate) (option|answer|response)\.?",
    r"determine the correct option:?",
    r"identify the correct statement:?",
    r"based on the given context\.?",
    r"which of the following( statements)?( is true)?",
]
_WS = re.compile(r'\s+')

def normalize_text(text):
    s = str(text)
    for pat in _BOILERPLATE:
        s = re.sub(pat, ' ', s, flags=re.IGNORECASE)
    return _WS.sub(' ', s).strip()

for df in (train, test):
    df['prompt_n'] = df['prompt'].map(normalize_text)
    for o in OPTS:
        df[f'{o}_n'] = df[o].map(normalize_text)

# Fold assignment (paraphrase-grouped)
gkf = GroupKFold(n_splits=CFG['N_FOLDS'])
train['fold'] = -1
for fold_id, (_, va_idx) in enumerate(gkf.split(train, groups=train['prompt_n'])):
    train.loc[train.index[va_idx], 'fold'] = fold_id
assert (train['fold'] >= 0).all()
print(f"Folds: {train['fold'].value_counts().sort_index().to_dict()}")

y_idx = np.array([OPTS.index(a) for a in train['answer']])
print(f"y_idx shape: {y_idx.shape}")


## 5 · LSTM vocabulary

The LSTM model embeds tokens by integer IDs. Vocab was built from train+test in the training
notebook — rebuilding here from the same CSVs produces an identical mapping.

In [ ]:
PAD, UNK = 0, 1

def build_vocab(texts, min_freq=2, max_size=30000):
    cnt = Counter()
    for t in texts:
        cnt.update(re.findall(r"[a-zA-Z']+|\d+", t.lower()))
    vocab = {'<pad>': PAD, '<unk>': UNK}
    for w, c in cnt.most_common(max_size):
        if c >= min_freq:
            vocab[w] = len(vocab)
    return vocab

def encode_text(text, vocab, max_len):
    toks = re.findall(r"[a-zA-Z']+|\d+", text.lower())[:max_len]
    ids  = [vocab.get(t, UNK) for t in toks]
    ids += [PAD] * (max_len - len(ids))
    return np.array(ids, dtype=np.int64)

_all = [train['prompt_n'].tolist(), test['prompt_n'].tolist()]
for o in OPTS:
    _all += [train[f'{o}_n'].tolist(), test[f'{o}_n'].tolist()]
VOCAB = build_vocab([t for grp in _all for t in grp])
print(f"LSTM vocab size: {len(VOCAB)}")
